# Phân loại Email Spam
## 3. Làm sạch dữ liệu

Các bước thực hiện:
1. Xử lý giá trị bị thiếu trong cột `Subject` và `Message`
2. Loại bỏ các dòng bị trùng lặp
3. Chuẩn hóa tên cột nhãn
4. Kết hợp tiêu đề và nội dung email thành một trường văn bản duy nhất
5. Lưu lại tập dữ liệu đã làm sạch


In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv(r'../data/raw/enron_spam_data.csv')
print('Kích thước ban đầu:', df.shape)

Kích thước ban đầu: (33716, 5)


### 3.1 Xử lý giá trị thiếu

In [2]:
# Kiểm tra các dòng bị khuyết thiếu cả Subject và Message (hoàn toàn không có dữ liệu văn bản)
both_missing = df['Subject'].isnull() & df['Message'].isnull()
n_both_missing = both_missing.sum()
print(f'Số dòng bị khuyết thiếu cả Subject và Message: {n_both_missing}')

# Loại bỏ các dòng bị khuyết thiếu cả hai
if n_both_missing > 0:
    df = df[~both_missing]
    print(f'Đã loại bỏ {n_both_missing} dòng trống hoàn toàn.')

# Điền giá trị chuỗi rỗng cho các dòng khuyết thiếu đơn lẻ để các xử lý văn bản đồng bộ
df['Subject'] = df['Subject'].fillna('')
df['Message'] = df['Message'].fillna('')

print('\nSố lượng giá trị thiếu sau khi làm sạch:')
print(df.isnull().sum())

Số dòng bị khuyết thiếu cả Subject và Message: 51
Đã loại bỏ 51 dòng trống hoàn toàn.

Số lượng giá trị thiếu sau khi làm sạch:
Message ID    0
Subject       0
Message       0
Spam/Ham      0
Date          0
dtype: int64


### 3.2 Loại bỏ các dòng trùng lặp

In [3]:
before = len(df)
df = df.drop_duplicates()
after  = len(df)
print(f'Số dòng trước: {before:,}  |  Số dòng sau: {after:,}  |  Đã loại bỏ: {before - after}')

Số dòng trước: 33,665  |  Số dòng sau: 33,665  |  Đã loại bỏ: 0


### 3.3 Chuẩn hóa cột nhãn

In [4]:
# Đổi tên cột nhãn và mã hóa nhị phân
df = df.rename(columns={'Spam/Ham': 'label'})
df['label_bin'] = df['label'].map({'spam': 1, 'ham': 0})

print('Phân phối nhãn sau khi làm sạch:')
print(df['label'].value_counts())
df.head(3)

Phân phối nhãn sau khi làm sạch:
label
spam    17120
ham     16545
Name: count, dtype: int64


,Message ID,Subject,Message,label,Date,label_bin
0,0,christmas tree farm pictures,,ham,1999-12-10,0
1,1,"vastar resources , inc .","gary , production from the high island larger ...",ham,1999-12-13,0
2,2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,ham,1999-12-14,0


### 3.4 Kết hợp Subject và Message

In [5]:
# Ghép tiêu đề và nội dung để sinh đặc trưng văn bản duy nhất
df['text'] = df['Subject'].str.strip() + ' ' + df['Message'].str.strip()
df['text'] = df['text'].str.strip()

print('Mẫu văn bản sau khi kết hợp (200 ký tự đầu):')
print(df['text'].iloc[0][:200])

Mẫu văn bản sau khi kết hợp (200 ký tự đầu):
christmas tree farm pictures


### 3.5 Lưu tập dữ liệu đã làm sạch

In [6]:
out_dir  = r'./data/preprocessed'
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, 'enron_cleaned.csv')

df[['Subject', 'Message', 'text', 'label', 'label_bin']].to_csv(out_path, index=False)
print('Đã lưu tập dữ liệu làm sạch tại:', out_path)
print('Kích thước cuối cùng:', df.shape)

Đã lưu tập dữ liệu làm sạch tại: ./data/preprocessed/enron_cleaned.csv
Kích thước cuối cùng: (33665, 7)
